# General Forecast, XGBoost and LightGBM on 500 SKUs

You can run this notebook in Jupyter.
Everything here has been run locally and takes about five minutes on a MacBook Pro's CPU once
General Forecast is installed, most of it training 500 LightGBM models.

We'll forecast four weeks of demand for the 500 best-selling products at one Walmart store
(CA_1) and compare General Forecast with XGBoost and LightGBM on the same rows. General Forecast gives each SKU
its own small table and forecasts all of them in one call, with no training. The tree models are
trained two ways, once across all SKUs with the SKU as a feature, and once per SKU. For the fit and predict API on a single SKU, see
[`sku_forecasting_quickstart.ipynb`](sku_forecasting_quickstart.ipynb).

## Install

To install General Forecast, follow the steps in our [quickstart](https://docs.generalforecast.com/get-started/quickstart/).
The cell below installs it from PyPI, along with the libraries this notebook uses.

In [1]:
%pip install -q generalforecast pandas lightgbm xgboost matplotlib

Note: you may need to restart the kernel to use updated packages.


## 1. The data, by SKU and week

In [2]:
import os
import time
import urllib.request
import zipfile

import lightgbm as lgb
import pandas as pd
import xgboost as xgb

import generalforecast as gf

In [3]:
if not os.path.exists("m5.zip"):
    urllib.request.urlretrieve(
        "https://github.com/Nixtla/m5-forecasts/raw/main/datasets/m5.zip", "m5.zip"
    )
with zipfile.ZipFile("m5.zip") as archive:
    sales = pd.read_csv(archive.open("sales_train_evaluation.csv"))
    calendar = pd.read_csv(archive.open("calendar.csv"), parse_dates=["date"])
    prices = pd.read_csv(archive.open("sell_prices.csv"))
# The calendar's rows are the days d_1, d_2, ... in order.
calendar.insert(0, "d", [f"d_{day}" for day in range(1, len(calendar) + 1)])

In [4]:
store = sales[sales["store_id"] == "CA_1"]
days = [column for column in store.columns if column.startswith("d_")]
top = store.set_index("item_id")[days].sum(axis=1).nlargest(500).index
daily = store[store["item_id"].isin(top)].melt(
    id_vars="item_id", value_vars=days, var_name="d", value_name="units"
)

In [5]:
calendar["event"] = calendar["event_name_1"].notna()
weeks = calendar.groupby("wm_yr_wk").agg(
    date=("date", "min"),
    days=("date", "size"),
    events=("event", "sum"),
    snap_days=("snap_CA", "sum"),
)
weeks = weeks[weeks["days"] == 7].drop(columns="days")

daily = daily.merge(calendar[["d", "wm_yr_wk"]], on="d")
weekly = daily.groupby(["item_id", "wm_yr_wk"])["units"].agg(["sum", "size"])
weekly = weekly[weekly["size"] == 7]["sum"].rename("units").reset_index()

In [6]:
grid = pd.MultiIndex.from_product([top, weeks.index], names=["item_id", "wm_yr_wk"])
data = grid.to_frame(index=False).merge(weeks, left_on="wm_yr_wk", right_index=True)
data = data.merge(weekly, how="left")
data = data.merge(
    prices[prices["store_id"] == "CA_1"][["item_id", "wm_yr_wk", "sell_price"]],
    how="left",
).rename(columns={"sell_price": "price"})
data = data.drop(columns="wm_yr_wk")
data[data["item_id"] == top[0]].tail(8)

,item_id,date,events,snap_days,units,price
273,FOODS_3_090,2016-04-23,0,0,384.0,1.6
274,FOODS_3_090,2016-04-30,3,6,473.0,1.6
275,FOODS_3_090,2016-05-07,1,4,441.0,1.6
276,FOODS_3_090,2016-05-14,0,0,360.0,1.6
277,FOODS_3_090,2016-05-21,0,0,NaN,1.6
278,FOODS_3_090,2016-05-28,2,3,NaN,1.6
279,FOODS_3_090,2016-06-04,1,7,NaN,1.6
280,FOODS_3_090,2016-06-11,0,0,NaN,1.6


## 2. Features

General Forecast reads `date` itself, as days since 1970, weekday, day of month and month, and the tree models
get those same four numbers. The tree models also get each SKU's sales in the same week a year
earlier, because a tree splits only on the columns of the row it predicts, and General Forecast reads each
SKU's whole table.

In [7]:
SERIES = "item_id"  # the column that names a SKU
DATE = "date"  # the week, as a date
TARGET = "units"  # what to forecast; empty in the weeks to forecast
KNOWN = ["price", "events", "snap_days"]  # columns known ahead of time
SEASON = 52  # periods in a year: 52 for weekly data, 12 for monthly
HORIZON = 4  # periods to forecast
HISTORY = 150  # periods of history per SKU

In [8]:
data = data.sort_values([SERIES, DATE])
data["last_year"] = data.groupby(SERIES)[TARGET].shift(SEASON)
FEATURES = [DATE, *KNOWN]

history = data[data[TARGET].notna()]
last = history[DATE].max()
print(
    f"{history[SERIES].nunique()} SKUs, {len(history):,} weeks of sales, up to {last:%Y-%m-%d}"
)

500 SKUs, 138,500 weeks of sales, up to 2016-05-14


## 3. Forecast the last four weeks

We hold out each SKU's last four weeks and give every model the 150 weeks before them. The score
is the mean absolute error in units per week, and the time is training plus predicting. General Forecast's
first prediction also downloads its weights and compiles.

In [9]:
cutoff = last - pd.Timedelta(weeks=HORIZON - 1)
test = history[history[DATE] >= cutoff]
train = history[
    (history[DATE] < cutoff) & (history[DATE] >= cutoff - pd.Timedelta(weeks=HISTORY))
]
results, forecasts = {}, {}


def score(name, started, forecast):
    """Keeps a forecast of the test weeks, at least 0 units, its mean absolute error and its time."""
    forecasts[name] = pd.Series(forecast, index=test.index).clip(lower=0)
    results[name] = {
        "MAE": (test[TARGET] - forecasts[name]).abs().mean(),
        "seconds": time.perf_counter() - started,
    }


# General Forecast: one regressor per SKU, each reading that SKU's weeks as its context.
started = time.perf_counter()
median = pd.Series(0.0, index=test.index)
for sku, rows in train.groupby(SERIES):
    ahead = test[test[SERIES] == sku]
    model = gf.Regressor("general-forecast/tabicl-v2-regressor")
    median.loc[ahead.index] = model.fit(rows[FEATURES], rows[TARGET]).predict(
        ahead[FEATURES]
    )
score("General Forecast", started, median)


def numbers(frame, with_sku=False):
    """The features as numbers for the tree models: the date as the four numbers General Forecast makes of it."""
    when = frame[DATE]
    X = frame[[*KNOWN, "last_year"]].assign(
        days=(when - pd.Timestamp("1970-01-01")).dt.days,
        weekday=when.dt.day_of_week,
        day=when.dt.day,
        month=when.dt.month,
    )
    return X.assign(sku=frame[SERIES].astype("category")) if with_sku else X


# XGBoost and LightGBM across all SKUs, with the SKU as a category.
for name, model in (
    (
        "XGBoost",
        xgb.XGBRegressor(
            objective="reg:absoluteerror",
            n_estimators=500,
            learning_rate=0.05,
            enable_categorical=True,
        ),
    ),
    (
        "LightGBM",
        lgb.LGBMRegressor(
            objective="regression_l1",
            n_estimators=500,
            learning_rate=0.05,
            verbosity=-1,
        ),
    ),
):
    started = time.perf_counter()
    model.fit(numbers(train, with_sku=True), train[TARGET])
    score(name, started, model.predict(numbers(test, with_sku=True)))

# LightGBM trained on each SKU alone.
started = time.perf_counter()
per_sku = pd.Series(0.0, index=test.index)
for sku, rows in train.groupby(SERIES):
    model = lgb.LGBMRegressor(
        objective="regression_l1", n_estimators=200, learning_rate=0.05, verbosity=-1
    )
    ahead = test[test[SERIES] == sku]
    per_sku.loc[ahead.index] = model.fit(numbers(rows), rows[TARGET]).predict(
        numbers(ahead)
    )
score("LightGBM per SKU", started, per_sku)

before = train.sort_values(DATE).groupby(SERIES)[TARGET]
score(
    "13-week average",
    time.perf_counter(),
    test[SERIES].map(before.apply(lambda units: units.tail(13).mean())),
)
score("same week last year", time.perf_counter(), test["last_year"])
print(pd.DataFrame(results).T.round(2).to_string())

                       MAE  seconds
General Forecast     10.81    72.89
XGBoost              11.19     6.29
LightGBM             11.46    47.96
LightGBM per SKU     11.83   217.96
13-week average      12.57     0.02
same week last year  18.48     0.00
